# LeetCode 3606 — Coupon Code Validator bằng Python

Link đề: https://leetcode.com/problems/coupon-code-validator/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def validateCoupons(self, code: List[str], businessLine: List[str], isActive: List[bool]) -> List[str]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lọc các coupon **hợp lệ** (mã không rỗng và chỉ gồm chữ/số/gạch dưới, `businessLine` thuộc 4 giá trị cho phép, `isActive` là `True`), rồi trả về **mã** theo thứ tự `businessLine` ưu tiên và **mã tăng dần** trong từng nhóm.
2. Dùng `zip` để duyệt song song ba mảng, và biết ba cách kiểm tra "chỉ gồm chữ, số, gạch dưới": duyệt từng ký tự, `str.isalnum`, `re.fullmatch`.
3. Sắp xếp theo **khóa ghép** `(độ ưu tiên, mã)` bằng `sorted(..., key=...)` và biết sắp xếp chuỗi theo thứ tự **ASCII** (chữ HOA đứng trước chữ thường).
4. Viết 4 cách: lọc + sắp xếp khóa ghép, regex + sắp xếp, gom theo nhóm (bucket) rồi sắp mỗi nhóm, và sắp xếp trước rồi lọc; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: quên mã rỗng, không lọc `isActive`, sắp xếp chỉ theo mã, sắp xếp không phân biệt hoa/thường, và `replace("_", "").isalnum()` làm sai mã chỉ gồm `_`.

## 2. Đề bài & ví dụ

Có `n` coupon, mô tả bởi ba mảng cùng độ dài `n`: `code[i]`, `businessLine[i]`, `isActive[i]`. Coupon `i` **hợp lệ** khi **cả ba** điều kiện đúng:

1. `code[i]` **không rỗng** và chỉ gồm chữ cái (`a-z`, `A-Z`), chữ số (`0-9`) và dấu gạch dưới `_`.
2. `businessLine[i]` là một trong `"electronics"`, `"grocery"`, `"pharmacy"`, `"restaurant"`.
3. `isActive[i]` là `true`.

Trả về danh sách **mã** của các coupon hợp lệ, **sắp xếp** theo `businessLine` theo thứ tự `electronics`, `grocery`, `pharmacy`, `restaurant`; trong cùng một `businessLine` thì sắp xếp **mã tăng dần theo thứ tự từ điển**.

| Ví dụ | code / businessLine / isActive | Output | Giải thích |
|---|---|---|---|
| 1 | `["SAVE20","","PHARMA5","SAVE@20"]` / `["restaurant","grocery","pharmacy","restaurant"]` / `[T,T,T,T]` | `["PHARMA5","SAVE20"]` | coupon 2 có mã rỗng; coupon 4 chứa `@`; `pharmacy` đứng trước `restaurant` |
| 2 | `["GROCERY15","ELECTRONICS_50","DISCOUNT10"]` / `["grocery","electronics","invalid"]` / `[F,T,T]` | `["ELECTRONICS_50"]` | coupon 1 không hoạt động; coupon 3 có `businessLine` không hợp lệ |

Constraints: `n == code.length == businessLine.length == isActive.length`, `1 <= n <= 100`, `0 <= code[i].length, businessLine[i].length <= 100`, `code[i]` và `businessLine[i]` gồm ký tự ASCII in được.

In [1]:
code = ["SAVE20", "", "PHARMA5", "SAVE@20"]
line = ["restaurant", "grocery", "pharmacy", "restaurant"]
active = [True, True, True, True]
for c, b, a in zip(code, line, active):
    print(repr(c), b, a)

'SAVE20' restaurant True
'' grocery True
'PHARMA5' pharmacy True
'SAVE@20' restaurant True


## 3. Kiến thức nền

### 3.1. `zip` để duyệt song song

`zip(a, b, c)` ghép phần tử cùng vị trí của nhiều danh sách: `(a[0], b[0], c[0])`, `(a[1], b[1], c[1])`, ... Nhờ đó không cần chỉ số `i`.

In [2]:
for c, b, a in zip(["x", "y"], ["g", "p"], [True, False]):
    print(c, b, a)

x g True
y p False


### 3.2. Kiểm tra ký tự hợp lệ

Có ba cách phổ biến để biết `code` chỉ gồm chữ, số, `_`:

- Duyệt từng ký tự `ch` và kiểm tra `ch.isalnum() or ch == "_"` (với ASCII thì `isalnum` đúng là chữ + số).
- Dùng tập ký tự cho phép: `set(code) <= ALLOWED`.
- Dùng biểu thức chính quy: `re.fullmatch(r"[A-Za-z0-9_]+", code)` (dấu `+` buộc ít nhất một ký tự, nên mã rỗng bị loại).

**Cạm bẫy:** `all(...)` trên chuỗi **rỗng** trả `True`, nên phải kiểm tra `code != ""` riêng.

In [3]:
import re
ALLOWED = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_")
for s in ["SAVE20", "", "SAVE@20", "A_1", "_"]:
    by_char = len(s) > 0 and all(ch.isalnum() or ch == "_" for ch in s)
    by_set = len(s) > 0 and set(s) <= ALLOWED
    by_re = re.fullmatch(r"[A-Za-z0-9_]+", s) is not None
    print(repr(s), by_char, by_set, by_re)
print(all([]), any([]))

'SAVE20' True True True
'' False False False
'SAVE@20' False False False
'A_1' True True True
'_' True True True
True False


### 3.3. Sắp xếp với khóa ghép

`sorted(items, key=f)` sắp xếp theo `f(item)`. Nếu `f` trả về **tuple** `(độ ưu tiên, mã)`, Python so sánh phần tử đầu trước, bằng nhau mới so phần tử sau. Chuỗi so sánh theo **mã ký tự**: `'0'..'9'` (48..57) < `'A'..'Z'` (65..90) < `'_'` (95) < `'a'..'z'` (97..122). Vậy `"B" < "_" < "a"`.

In [4]:
ORDER = {"electronics": 0, "grocery": 1, "pharmacy": 2, "restaurant": 3}
items = [("restaurant", "SAVE20"), ("pharmacy", "PHARMA5"), ("electronics", "b"), ("electronics", "B"), ("electronics", "_x")]
print(sorted(items, key=lambda it: (ORDER[it[0]], it[1])))
print(sorted(["b", "B", "_", "a", "9"]))
print(ord("B"), ord("_"), ord("a"))

[('electronics', 'B'), ('electronics', '_x'), ('electronics', 'b'), ('pharmacy', 'PHARMA5'), ('restaurant', 'SAVE20')]
['9', 'B', '_', 'a', 'b']
66 95 97


### 3.4. Từ điển ưu tiên và `in`

Dùng dict `ORDER = {"electronics": 0, ...}`: `b in ORDER` kiểm tra thuộc 4 giá trị hợp lệ (`O(1)`), và `ORDER[b]` cho luôn độ ưu tiên. Lưu ý phân biệt **hoa/thường**: `"Electronics"` **không** hợp lệ.

In [5]:
print("grocery" in ORDER, "Grocery" in ORDER, "" in ORDER, "invalid" in ORDER)
print(ORDER["pharmacy"])

True False False False
2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay với ví dụ 1:

```
coupon 1: "SAVE20", restaurant, True   -> hợp lệ
coupon 2: "",       grocery,    True   -> mã rỗng -> loại
coupon 3: "PHARMA5", pharmacy,  True   -> hợp lệ
coupon 4: "SAVE@20", restaurant, True  -> có '@' -> loại
giữ: (restaurant, SAVE20), (pharmacy, PHARMA5)
sắp theo (ưu tiên, mã): pharmacy(2) < restaurant(3) -> ["PHARMA5", "SAVE20"]
```

Hai pha: **lọc** (3 điều kiện) rồi **sắp xếp** (khóa ghép). Thứ tự pha không đổi kết quả (lọc trước nhanh hơn vì ít phần tử cần sắp).

## 5. Cách 1 — Brute Force / baseline

Duyệt từng coupon; kiểm tra mã bằng một vòng lặp ký tự thủ công; nếu hợp lệ thì thêm `(ưu tiên, mã)` vào danh sách; cuối cùng `sort()` rồi lấy phần mã.

Nút thắt: không có (với `n <= 100`). Điểm cần chú ý chỉ là **đúng tất cả điều kiện**; cách này viết dài nhưng rất dễ kiểm chứng.

In [6]:
class SolutionBrute:
    def validateCoupons(self, code, businessLine, isActive):
        order = {"electronics": 0, "grocery": 1, "pharmacy": 2, "restaurant": 3}
        good = []
        for i in range(len(code)):
            if not isActive[i]:
                continue
            if businessLine[i] not in order:
                continue
            if len(code[i]) == 0:
                continue
            ok = True
            for ch in code[i]:
                if not (ch.isalnum() or ch == "_"):
                    ok = False
                    break
            if ok:
                good.append((order[businessLine[i]], code[i]))
        good.sort()
        return [c for _, c in good]

code = ["SAVE20", "", "PHARMA5", "SAVE@20"]
line = ["restaurant", "grocery", "pharmacy", "restaurant"]
print(SolutionBrute().validateCoupons(code, line, [True] * 4))
print(SolutionBrute().validateCoupons(["GROCERY15", "ELECTRONICS_50", "DISCOUNT10"], ["grocery", "electronics", "invalid"], [False, True, True]))

['PHARMA5', 'SAVE20']
['ELECTRONICS_50']


**Độ phức tạp:** `L` = tổng độ dài các mã (`<= 100 * 100`). Lọc `O(L)`; sắp xếp `O(m log m)` phép so sánh, mỗi phép so sánh chuỗi tối đa `O(100)`; Space `O(m)` với `m` = số coupon hợp lệ.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** gói "ưu tiên nhóm" và "mã" thành **một khóa so sánh duy nhất** `(ORDER[businessLine], code)`. Khi đó cả yêu cầu sắp xếp hai tầng chỉ cần **một** lần `sorted`.

**Quy tắc "nếu… thì…":**

- Nếu `isActive[i]` sai -> loại.
- Nếu `businessLine[i]` không nằm trong `ORDER` -> loại.
- Nếu `code[i]` rỗng hoặc có ký tự ngoài `[A-Za-z0-9_]` -> loại.
- Còn lại -> giữ, khóa sắp xếp là `(ORDER[b], c)`.

Invariant: sau khi lọc, mọi phần tử còn lại thỏa cả 3 điều kiện; sau `sorted`, đầu ra không giảm theo khóa ghép.

In [7]:
def valid_codes(code, line, active):
    keep = [(ORDER[b], c) for c, b, a in zip(code, line, active)
            if a and b in ORDER and c != "" and all(ch.isalnum() or ch == "_" for ch in c)]
    print("sau khi lọc:", keep)
    return [c for _, c in sorted(keep)]

print(valid_codes(["SAVE20", "", "PHARMA5", "SAVE@20"], ["restaurant", "grocery", "pharmacy", "restaurant"], [True] * 4))

sau khi lọc: [(3, 'SAVE20'), (2, 'PHARMA5')]
['PHARMA5', 'SAVE20']


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) vòng lặp ký tự thủ công + `sort`; (2) regex `fullmatch` + `sorted(key=...)`; (3) **bucket** theo `businessLine`, sắp mỗi nhóm; (4) sắp xếp **chỉ số** theo khóa rồi lọc.

### Cách 2 — Regex + `sorted(key=...)`

Dùng `re.fullmatch(r"[A-Za-z0-9_]+", code)`: toàn bộ chuỗi phải khớp và ít nhất một ký tự, nên mã rỗng tự bị loại. Sau đó `sorted` với `key=lambda p: (ORDER[p[0]], p[1])`.

In [8]:
import re

class SolutionRegex:
    PATTERN = re.compile(r"[A-Za-z0-9_]+")
    ORDER = {"electronics": 0, "grocery": 1, "pharmacy": 2, "restaurant": 3}

    def validateCoupons(self, code, businessLine, isActive):
        valid = [
            (b, c)
            for c, b, a in zip(code, businessLine, isActive)
            if a and b in self.ORDER and self.PATTERN.fullmatch(c)
        ]
        valid.sort(key=lambda p: (self.ORDER[p[0]], p[1]))
        return [c for _, c in valid]

print(SolutionRegex().validateCoupons(["SAVE20", "", "PHARMA5", "SAVE@20"], ["restaurant", "grocery", "pharmacy", "restaurant"], [True] * 4))
print(bool(re.fullmatch(r"[A-Za-z0-9_]+", "")), bool(re.fullmatch(r"[A-Za-z0-9_]+", "a_1")))

['PHARMA5', 'SAVE20']
False True


Độ phức tạp Cách 2: giống Cách 1 — Time `O(L + m log m)`, Space `O(m)`.

### Cách 3 — Bucket theo `businessLine`

Tạo 4 "xô" theo thứ tự ưu tiên. Mã hợp lệ bỏ vào xô của `businessLine` nó; cuối cùng sắp **từng xô** rồi **nối** các xô theo thứ tự. Không cần khóa ghép vì thứ tự nhóm do thứ tự xô quyết định.

In [9]:
class SolutionBucket:
    LINES = ["electronics", "grocery", "pharmacy", "restaurant"]

    def validateCoupons(self, code, businessLine, isActive):
        buckets = {b: [] for b in self.LINES}
        allowed = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_")
        for c, b, a in zip(code, businessLine, isActive):
            if a and b in buckets and c and set(c) <= allowed:
                buckets[b].append(c)
        out = []
        for b in self.LINES:
            buckets[b].sort()
            out.extend(buckets[b])
        return out

print(SolutionBucket().validateCoupons(["SAVE20", "", "PHARMA5", "SAVE@20"], ["restaurant", "grocery", "pharmacy", "restaurant"], [True] * 4))
print(SolutionBucket().validateCoupons(["GROCERY15", "ELECTRONICS_50", "DISCOUNT10"], ["grocery", "electronics", "invalid"], [False, True, True]))

['PHARMA5', 'SAVE20']
['ELECTRONICS_50']


Độ phức tạp Cách 3: Time `O(L + m log m)` (mỗi xô được sắp riêng nên tổng so sánh không lớn hơn sắp chung), Space `O(m)`.

### Cách 4 — Sắp xếp chỉ số rồi lọc

Sắp xếp **mọi** chỉ số `0..n-1` theo khóa `(ưu tiên, mã)` (với `businessLine` lạ cho ưu tiên rất lớn), rồi duyệt theo thứ tự đó và chỉ lấy coupon hợp lệ. Ý tưởng "giữ chỉ số rồi tra ngược" hữu ích khi cần trả về **chỉ số** thay vì giá trị.

In [10]:
class SolutionSortIdx:
    def validateCoupons(self, code, businessLine, isActive):
        order = {"electronics": 0, "grocery": 1, "pharmacy": 2, "restaurant": 3}
        idx = sorted(range(len(code)), key=lambda i: (order.get(businessLine[i], 99), code[i]))
        res = []
        for i in idx:
            c = code[i]
            if isActive[i] and businessLine[i] in order and c != "" and all(ch.isalnum() or ch == "_" for ch in c):
                res.append(c)
        return res

print(SolutionSortIdx().validateCoupons(["SAVE20", "", "PHARMA5", "SAVE@20"], ["restaurant", "grocery", "pharmacy", "restaurant"], [True] * 4))
print(SolutionSortIdx().validateCoupons(["b", "B"], ["grocery", "grocery"], [True, True]))

['PHARMA5', 'SAVE20']
['B', 'b']


Độ phức tạp Cách 4: Time `O(n log n)` (sắp cả những coupon sẽ bị loại), Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 1 trên ví dụ 1 (`order: electronics=0, grocery=1, pharmacy=2, restaurant=3`):

| i | code | businessLine | active | kết quả kiểm tra | đưa vào `good`? |
|---|---|---|---|---|---|
| 0 | `SAVE20` | restaurant | True | đủ 3 điều kiện | `(3, "SAVE20")` |
| 1 | `""` | grocery | True | mã rỗng | không |
| 2 | `PHARMA5` | pharmacy | True | đủ 3 điều kiện | `(2, "PHARMA5")` |
| 3 | `SAVE@20` | restaurant | True | `@` không hợp lệ | không |

Sắp `good` -> `[(2,"PHARMA5"), (3,"SAVE20")]` -> trả `["PHARMA5", "SAVE20"]`.

In [11]:
def trace_validate(code, line, active):
    order = {"electronics": 0, "grocery": 1, "pharmacy": 2, "restaurant": 3}
    good = []
    for i, (c, b, a) in enumerate(zip(code, line, active)):
        if not a:
            print(f"i={i} {c!r}: không hoạt động -> loại")
            continue
        if b not in order:
            print(f"i={i} {c!r}: businessLine {b!r} không hợp lệ -> loại")
            continue
        if c == "":
            print(f"i={i} {c!r}: mã rỗng -> loại")
            continue
        bad = [ch for ch in c if not (ch.isalnum() or ch == "_")]
        if bad:
            print(f"i={i} {c!r}: ký tự không hợp lệ {bad} -> loại")
            continue
        good.append((order[b], c))
        print(f"i={i} {c!r}: hợp lệ, khóa = {(order[b], c)}")
    good.sort()
    print("sau khi sắp:", good)
    return [c for _, c in good]

print(trace_validate(["SAVE20", "", "PHARMA5", "SAVE@20"], ["restaurant", "grocery", "pharmacy", "restaurant"], [True] * 4))
print(trace_validate(["GROCERY15", "ELECTRONICS_50", "DISCOUNT10"], ["grocery", "electronics", "invalid"], [False, True, True]))

i=0 'SAVE20': hợp lệ, khóa = (3, 'SAVE20')
i=1 '': mã rỗng -> loại
i=2 'PHARMA5': hợp lệ, khóa = (2, 'PHARMA5')
i=3 'SAVE@20': ký tự không hợp lệ ['@'] -> loại
sau khi sắp: [(2, 'PHARMA5'), (3, 'SAVE20')]
['PHARMA5', 'SAVE20']
i=0 'GROCERY15': không hoạt động -> loại
i=1 'ELECTRONICS_50': hợp lệ, khóa = (0, 'ELECTRONICS_50')
i=2 'DISCOUNT10': businessLine 'invalid' không hợp lệ -> loại
sau khi sắp: [(0, 'ELECTRONICS_50')]
['ELECTRONICS_50']


## 9. Code nộp LeetCode

In [12]:
from typing import List

class Solution:
    def validateCoupons(self, code: List[str], businessLine: List[str], isActive: List[bool]) -> List[str]:
        order = {"electronics": 0, "grocery": 1, "pharmacy": 2, "restaurant": 3}
        good = []
        for c, b, a in zip(code, businessLine, isActive):
            if not a or b not in order or c == "":
                continue
            if all(ch.isalnum() or ch == "_" for ch in c):
                good.append((order[b], c))
        good.sort()
        return [c for _, c in good]

print(Solution().validateCoupons(["SAVE20", "", "PHARMA5", "SAVE@20"], ["restaurant", "grocery", "pharmacy", "restaurant"], [True, True, True, True]))
print(Solution().validateCoupons(["GROCERY15", "ELECTRONICS_50", "DISCOUNT10"], ["grocery", "electronics", "invalid"], [False, True, True]))

['PHARMA5', 'SAVE20']
['ELECTRONICS_50']


## 10. Vì sao thuật toán đúng

- **Bộ lọc đúng đề:** mỗi coupon bị loại khi vi phạm ít nhất một trong ba điều kiện; coupon còn lại thỏa cả ba. Mã rỗng bị loại riêng nên không lọt qua `all(...)` trên chuỗi rỗng.
- **Sắp xếp hai tầng bằng khóa ghép:** so sánh tuple `(p1, c1) < (p2, c2)` xảy ra khi `p1 < p2`, hoặc `p1 == p2` và `c1 < c2`. Đây chính xác là "theo `businessLine` ưu tiên, trong nhóm theo mã tăng dần".
- **Thứ tự từ điển = thứ tự mã ký tự:** Python so chuỗi theo từng mã ký tự, trùng với "lexicographical order" của đề cho ký tự ASCII (đó là lý do `"B" < "a"`).
- **Cách 3 (bucket):** thứ tự các xô cố định là thứ tự ưu tiên; sắp xếp từng xô cho thứ tự mã; nối lại cho đúng thứ tự hai tầng.
- **Cách 4:** lọc theo thứ tự đã sắp vẫn giữ nguyên thứ tự tương đối, nên kết quả bằng "lọc rồi sắp".
- **Trùng mã:** hai coupon có thể cùng mã; mỗi coupon hợp lệ vẫn xuất hiện, đúng đề (không loại trùng).

## 11. Phân tích Time / Space Complexity

`n` = số coupon, `m` = số coupon hợp lệ, `L` = tổng độ dài các mã.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp ký tự + `sort` | `O(L + m log m)` | `O(m)` |
| 2. Regex + `sorted(key)` | `O(L + m log m)` | `O(m)` |
| 3. Bucket + sắp từng xô | `O(L + m log m)` | `O(m)` |
| 4. Sắp chỉ số rồi lọc | `O(n log n + L)` | `O(n)` |

Mỗi phép so sánh hai chuỗi tốn tối đa `O(100)`. Với `n <= 100` tất cả đều tức thì.

## 12. Bộ test

Test 1: ví dụ của đề và các edge case (đều tuân constraints: `1 <= n <= 100`, độ dài chuỗi `<= 100`, ASCII in được).

In [13]:
impls = [SolutionBrute(), SolutionRegex(), SolutionBucket(), SolutionSortIdx(), Solution()]

def oracle(code, line, active):
    allowed = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_")
    out = []
    for rank, name in enumerate(["electronics", "grocery", "pharmacy", "restaurant"]):
        mine = []
        for i in range(len(code)):
            if active[i] and line[i] == name and len(code[i]) > 0 and set(code[i]).issubset(allowed):
                mine.append(code[i])
        out += sorted(mine)
    return out

tests = [
    (["SAVE20", "", "PHARMA5", "SAVE@20"], ["restaurant", "grocery", "pharmacy", "restaurant"], [True] * 4, ["PHARMA5", "SAVE20"]),
    (["GROCERY15", "ELECTRONICS_50", "DISCOUNT10"], ["grocery", "electronics", "invalid"], [False, True, True], ["ELECTRONICS_50"]),
    (["a"], ["grocery"], [True], ["a"]),
    ([""], ["grocery"], [True], []),                                   # mã rỗng
    (["_"], ["electronics"], [True], ["_"]),                           # chỉ gạch dưới là hợp lệ
    (["b", "B", "a", "A"], ["grocery"] * 4, [True] * 4, ["A", "B", "a", "b"]),     # thứ tự ASCII
    (["x", "x"], ["pharmacy", "pharmacy"], [True, True], ["x", "x"]),   # trùng mã vẫn giữ cả hai
    (["c1", "c2", "c3", "c4"], ["restaurant", "pharmacy", "grocery", "electronics"], [True] * 4, ["c4", "c3", "c2", "c1"]),
    (["ok"], ["Grocery"], [True], []),                                 # phân biệt hoa/thường
    (["a b"], ["grocery"], [True], []),                                # dấu cách không hợp lệ
    (["a-b", "a_b"], ["grocery", "grocery"], [True, True], ["a_b"]),
    (["z", "y"], ["electronics", "electronics"], [False, False], []),   # đều không hoạt động
    (["A1"], [""], [True], []),                                        # businessLine rỗng
]
for code, line, active, exp in tests:
    assert 1 <= len(code) <= 100 and len(code) == len(line) == len(active)
    assert all(len(s) <= 100 for s in code + line)
    assert oracle(code, line, active) == exp, (code, oracle(code, line, active), exp)
    for im in impls:
        assert im.validateCoupons(list(code), list(line), list(active)) == exp, (type(im).__name__, code)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 13


Test 2: ngẫu nhiên có seed, đúng constraints. Mã sinh từ bảng ký tự gồm cả ký tự hợp lệ và không hợp lệ (`@ - $ khoảng trắng`), `businessLine` gồm 4 giá trị đúng, giá trị sai, sai hoa/thường và rỗng.

In [14]:
import random
random.seed(3606)

VALID_CH = "abcABC019_"
BAD_CH = "@-$ !"
LINES = ["electronics", "grocery", "pharmacy", "restaurant", "invalid", "", "Grocery", "foods"]

def rand_code():
    L = random.choice([0, 1, 1, 2, 3, 5, 8])
    if random.random() < 0.7:
        return "".join(random.choice(VALID_CH) for _ in range(L))
    return "".join(random.choice(VALID_CH + BAD_CH) for _ in range(L))

nonempty = 0
for t in range(3000):
    n = random.randint(1, 30)
    code = [rand_code() for _ in range(n)]
    line = [random.choice(LINES) for _ in range(n)]
    active = [random.random() < 0.8 for _ in range(n)]
    assert all(len(s) <= 100 for s in code + line)
    exp = oracle(code, line, active)
    nonempty += bool(exp)
    for im in impls:
        assert im.validateCoupons(list(code), list(line), list(active)) == exp, (type(im).__name__, code, line, active)
print("3000 test ngẫu nhiên OK; số ca có kết quả không rỗng =", nonempty)

3000 test ngẫu nhiên OK; số ca có kết quả không rỗng = 2741


Test 3: tính chất — đầu ra là **tập con có bội** của `code`; đầu vào không bị sửa; xáo trộn thứ tự coupon **không** đổi đầu ra (vì đầu ra được sắp xếp hoàn toàn); và đầu ra không giảm theo khóa ghép.

In [15]:
from collections import Counter
random.seed(606)
ORD = {"electronics": 0, "grocery": 1, "pharmacy": 2, "restaurant": 3}
for _ in range(1000):
    n = random.randint(1, 40)
    code = [rand_code() for _ in range(n)]
    line = [random.choice(LINES) for _ in range(n)]
    active = [random.random() < 0.8 for _ in range(n)]
    snapshot = (list(code), list(line), list(active))
    res = Solution().validateCoupons(code, line, active)
    assert (code, line, active) == snapshot
    assert not (Counter(res) - Counter(code))
    perm = list(range(n))
    random.shuffle(perm)
    res2 = Solution().validateCoupons([code[i] for i in perm], [line[i] for i in perm], [active[i] for i in perm])
    assert res2 == res
print("tính chất (tập con, không sửa input, bất biến với hoán vị): OK")

tính chất (tập con, không sửa input, bất biến với hoán vị): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên mã rỗng (`all([])` là `True`)

In [16]:
def wrong_empty(code, line, active):
    good = []
    for c, b, a in zip(code, line, active):
        if a and b in ORDER and all(ch.isalnum() or ch == "_" for ch in c):     # SAI: thiếu c != ""
            good.append((ORDER[b], c))
    return [c for _, c in sorted(good)]

args = ([""], ["grocery"], [True])
print("sai :", wrong_empty(*args))
print("đúng:", Solution().validateCoupons(*args))
assert wrong_empty(*args) != Solution().validateCoupons(*args)

sai : ['']
đúng: []


Vì sao sai: `all(...)` trên chuỗi rỗng là `True` nên mã rỗng lọt qua. Đề yêu cầu mã **không rỗng**. **Sửa:** thêm `c != ""` (hoặc dùng regex có `+`).

### Lỗi 2: sắp xếp chỉ theo mã, bỏ qua thứ tự `businessLine`

In [17]:
def wrong_sort_code_only(code, line, active):
    good = []
    for c, b, a in zip(code, line, active):
        if a and b in ORDER and c != "" and all(ch.isalnum() or ch == "_" for ch in c):
            good.append(c)
    return sorted(good)                          # SAI: không xét nhóm

args = (["A", "B"], ["restaurant", "electronics"], [True, True])
print("sai :", wrong_sort_code_only(*args))
print("đúng:", Solution().validateCoupons(*args))
assert wrong_sort_code_only(*args) == ["A", "B"]
assert Solution().validateCoupons(*args) == ["B", "A"]

sai : ['A', 'B']
đúng: ['B', 'A']


Vì sao sai: mã `"B"` thuộc `electronics` (ưu tiên 0) phải đứng trước mã `"A"` thuộc `restaurant` (ưu tiên 3) dù `"A" < "B"`. **Sửa:** khóa ghép `(ORDER[b], c)`.

### Lỗi 3: bỏ qua `isActive`

In [18]:
def wrong_ignore_active(code, line, active):
    good = []
    for c, b, a in zip(code, line, active):
        if b in ORDER and c != "" and all(ch.isalnum() or ch == "_" for ch in c):    # SAI: thiếu `a`
            good.append((ORDER[b], c))
    return [c for _, c in sorted(good)]

args = (["GROCERY15", "ELECTRONICS_50", "DISCOUNT10"], ["grocery", "electronics", "invalid"], [False, True, True])
print("sai :", wrong_ignore_active(*args))
print("đúng:", Solution().validateCoupons(*args))
assert wrong_ignore_active(*args) != Solution().validateCoupons(*args)

sai : ['ELECTRONICS_50', 'GROCERY15']
đúng: ['ELECTRONICS_50']


Vì sao sai: coupon `GROCERY15` không hoạt động nhưng bị đưa vào kết quả. **Sửa:** kiểm tra `isActive[i]`.

### Lỗi 4: sắp xếp không phân biệt hoa/thường

In [19]:
def wrong_casefold(code, line, active):
    good = []
    for c, b, a in zip(code, line, active):
        if a and b in ORDER and c != "" and all(ch.isalnum() or ch == "_" for ch in c):
            good.append((ORDER[b], c))
    return [c for _, c in sorted(good, key=lambda p: (p[0], p[1].lower()))]     # SAI: dùng lower()

args = (["b", "B"], ["grocery", "grocery"], [True, True])
print("sai :", wrong_casefold(*args))
print("đúng:", Solution().validateCoupons(*args))
assert wrong_casefold(*args) != Solution().validateCoupons(*args)

sai : ['b', 'B']
đúng: ['B', 'b']


Vì sao sai: đề dùng thứ tự từ điển theo mã ký tự, nên `"B"` (66) đứng trước `"b"` (98). Với `lower()` hai khóa bằng nhau và Python giữ thứ tự vào `["b", "B"]`. **Sửa:** sắp theo chuỗi gốc.

### Lỗi 5: `replace("_", "").isalnum()` làm sai mã chỉ gồm `_`

In [20]:
def wrong_replace(code, line, active):
    good = []
    for c, b, a in zip(code, line, active):
        if a and b in ORDER and c.replace("_", "").isalnum():      # SAI: "_".replace -> "" -> isalnum() False
            good.append((ORDER[b], c))
    return [c for _, c in sorted(good)]

args = (["_", "A_"], ["grocery", "grocery"], [True, True])
print("sai :", wrong_replace(*args))
print("đúng:", Solution().validateCoupons(*args))
assert wrong_replace(*args) != Solution().validateCoupons(*args)

sai : ['A_']
đúng: ['A_', '_']


Vì sao sai: `"_"` sau khi bỏ gạch dưới thành `""`, và `"".isalnum()` là `False` nên mã `"_"` (hợp lệ) bị loại nhầm. **Sửa:** kiểm tra từng ký tự `ch.isalnum() or ch == "_"` hoặc dùng regex.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lọc rồi sắp xếp theo nhiều tiêu chí" -> `sorted(..., key=lambda x: (tiêu chí 1, tiêu chí 2))`.
- Thứ tự tùy ý của một tập nhãn (không alphabet) -> dict `ORDER` ánh xạ nhãn -> hạng; `b in ORDER` kiểm tra hợp lệ luôn.
- Chuỗi rỗng và `all([])`: luôn xét riêng hoặc dùng regex `+`.
- `str.isalnum()` đúng cho ASCII; với Unicode nó cũng nhận chữ có dấu, nhưng đề đảm bảo ASCII nên an toàn.
- Python so sánh chuỗi theo mã ký tự: chữ HOA < `_` < chữ thường.
- Viết test với trường hợp "gần đúng" (`Grocery`, `a b`, `a-b`) để bắt lỗi lọc.

## 15. Biến thể của bài

1. Trả về **chỉ số** (hoặc cả cặp `(code, businessLine)`) thay vì chỉ mã.
2. Thứ tự `businessLine` cho trước bằng một **danh sách khác** (tham số) thay vì cố định.
3. Mã phải **bắt đầu bằng chữ cái** và có độ dài tối thiểu `k`.
4. Đếm số coupon hợp lệ theo từng `businessLine` (dùng `Counter`).
5. Sắp xếp **giảm dần** theo mã trong từng nhóm (đảo khóa thứ hai, hoặc `sort(reverse=True)` từng xô).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp ký tự + `sort` | kiểm tra thủ công từng điều kiện | `O(L + m log m)` | `O(m)` | dễ debug, baseline |
| 2. Regex + `sorted(key)` | `fullmatch` + khóa ghép | `O(L + m log m)` | `O(m)` | ngắn gọn khi quen regex |
| 3. Bucket theo nhóm | 4 xô, sắp từng xô, nối | `O(L + m log m)` | `O(m)` | nhiều nhóm cố định, không cần khóa ghép |
| 4. Sắp chỉ số rồi lọc | sắp mọi chỉ số, lọc sau | `O(n log n + L)` | `O(n)` | khi cần trả về chỉ số |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().validateCoupons(["B1", "A1"], ["grocery", "grocery"], [True, True])` là gì?

**Bài 2 (tính tay):** `code = ["x_1", "y!", "z"]`, `businessLine = ["pharmacy", "grocery", "electronics"]`, `isActive = [True, True, False]`. Đáp án là gì?

**Bài 3 (điền TODO):** hoàn thành `is_valid_code(code)` trả `True` nếu mã không rỗng và chỉ gồm chữ/số/`_`.

**Bài 4 (tự viết lại):** viết `count_by_line(code, line, active)` trả về dict `{businessLine: số coupon hợp lệ}` cho 4 nhóm (nhóm không có thì `0`).

**Bài 5 (mini challenge):** viết `validate_indices(code, line, active)` trả về **danh sách chỉ số** coupon hợp lệ theo thứ tự đã sắp (cùng quy tắc với bài gốc; nếu bằng nhau về khóa thì chỉ số nhỏ trước).

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().validateCoupons(["B1", "A1"], ["grocery", "grocery"], [True, True]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def is_valid_code(code):
    # TODO: trả True nếu code != "" và mọi ký tự là chữ/số/_
    return None

['A1', 'B1']


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** lọc trước (3 điều kiện), sắp xếp sau. Nhớ quy tắc thứ tự ASCII: chữ số < HOA < `_` < thường.

**Mức 2 (vừa):** Bài 3: `len(code) > 0 and all(ch.isalnum() or ch == "_" for ch in code)`. Bài 4: tạo dict 4 nhóm bằng `0` rồi cộng `1` cho coupon hợp lệ. Bài 5: `sorted(valid_indices, key=lambda i: (ORDER[line[i]], code[i], i))`.

**Mức 3 (gần đáp án):** Bài 1: `"A1" < "B1"` -> `["A1", "B1"]`. Bài 2: coupon 2 có `!`, coupon 3 không hoạt động -> chỉ `"x_1"`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
assert Solution().validateCoupons(["B1", "A1"], ["grocery", "grocery"], [True, True]) == ["A1", "B1"]

# Bài 2
assert Solution().validateCoupons(["x_1", "y!", "z"], ["pharmacy", "grocery", "electronics"], [True, True, False]) == ["x_1"]

# Bài 3
def is_valid_code(code):
    return len(code) > 0 and all(ch.isalnum() or ch == "_" for ch in code)
assert is_valid_code("A_1") is True
assert is_valid_code("") is False
assert is_valid_code("_") is True
assert is_valid_code("A-1") is False

# Bài 4
def count_by_line(code, line, active):
    res = {b: 0 for b in ORDER}
    for c, b, a in zip(code, line, active):
        if a and b in ORDER and is_valid_code(c):
            res[b] += 1
    return res
assert count_by_line(["a", "b", "c"], ["grocery", "grocery", "pharmacy"], [True, False, True]) == {"electronics": 0, "grocery": 1, "pharmacy": 1, "restaurant": 0}

# Bài 5
def validate_indices(code, line, active):
    ids = [i for i in range(len(code)) if active[i] and line[i] in ORDER and is_valid_code(code[i])]
    return sorted(ids, key=lambda i: (ORDER[line[i]], code[i], i))
code5, line5, act5 = ["SAVE20", "", "PHARMA5", "SAVE@20"], ["restaurant", "grocery", "pharmacy", "restaurant"], [True] * 4
assert validate_indices(code5, line5, act5) == [2, 0]
assert [code5[i] for i in validate_indices(code5, line5, act5)] == Solution().validateCoupons(code5, line5, act5)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lọc rồi sắp xếp theo khóa ghép":**

```python
ORDER = {"electronics": 0, "grocery": 1, "pharmacy": 2, "restaurant": 3}
good = []
for c, b, a in zip(code, businessLine, isActive):
    if a and b in ORDER and c != "" and all(ch.isalnum() or ch == "_" for ch in c):
        good.append((ORDER[b], c))
good.sort()
return [c for _, c in good]
```

**Checklist trước khi submit:**

- [ ] Đủ **3 điều kiện**: mã hợp lệ và không rỗng, `businessLine` thuộc 4 giá trị, `isActive`?
- [ ] Mã rỗng bị loại (không phụ thuộc `all([])`)?
- [ ] Sắp theo `(ưu tiên nhóm, mã)`, không sắp riêng theo mã hay theo `lower()`?
- [ ] Thứ tự nhóm đúng: `electronics`, `grocery`, `pharmacy`, `restaurant`?
- [ ] Trả về **mã** (không phải cả cặp), giữ trùng mã?
- [ ] Đã thử mã `"_"`, `"a b"`, `"a-b"`, `"Grocery"`, `"b"` vs `"B"`?
- [ ] Chữ ký đúng `validateCoupons(self, code, businessLine, isActive) -> List[str]`?